# Training Dynamics: Vanishing Gradients, LayerNorm, Mixed Precision

Companion notebook for the [Training Dynamics](https://codemia.io/courses/deep_learning_foundations/training_dynamics) lesson on Codemia.

**What you will build**: a 10-layer MLP that you watch *fail* from vanishing gradients, then *heal* once you add LayerNorm, then *speed up* once you enable mixed precision.

**Karpathy companion**: [makemore Part 3: Activations, Gradients, BatchNorm](https://www.youtube.com/watch?v=P6sfmUTpUmc). This notebook is a direct spiritual successor — same diagnostic, different normalization.

*Adapted in part from karpathy/makemore (MIT) — the per-layer gradient histogram diagnostic is lifted from that video.*

**Time**: ~45 minutes  
**GPU required**: recommended for the mixed-precision section (otherwise the speedup is meaningless); CPU is fine for the rest.

## Setup

In [ ]:
%pip install -q torch matplotlib

In [ ]:
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using {device}')
if device == 'cuda':
    print('  GPU:', torch.cuda.get_device_name(0))

## Concept recap

Deep networks have a dirty secret: gradients don't make it cleanly from the loss back to the early layers. Each layer multiplies the upstream gradient by its Jacobian; if those Jacobians are consistently smaller than 1, the gradient shrinks exponentially with depth. By the time it hits layer 1, it is noise.

We will:
1. Build a deep MLP with a bad-but-realistic init and watch layer-1 gradients evaporate.
2. Drop in `nn.LayerNorm` between every linear+activation and watch the gradient magnitudes recover.
3. Enable `torch.amp` autocast + GradScaler and time both forward/backward passes.

## Implementation

### Step 1 — Synthetic dataset

We use a synthetic two-Gaussian-blob classification task. This lets the notebook run with zero downloads and keeps compute trivial so we can focus on the diagnostic, not the accuracy.

In [ ]:
def make_dataset(n=4000, d=64):
    # Two Gaussian clusters centered at +mu and -mu. Linearly separable in principle, but
    # a 10-layer MLP still has to actually propagate gradients to learn the decision boundary.
    mu = torch.randn(d) * 0.5
    x0 = torch.randn(n // 2, d) + mu
    x1 = torch.randn(n // 2, d) - mu
    X = torch.cat([x0, x1], dim=0)
    y = torch.cat([torch.zeros(n // 2), torch.ones(n // 2)]).long()
    # Shuffle so minibatches are mixed.
    idx = torch.randperm(n)
    return X[idx].to(device), y[idx].to(device)

X, y = make_dataset(n=4000, d=64)
print('X:', X.shape, 'y:', y.shape, 'class balance:', y.float().mean().item())

### Step 2 — A 10-layer MLP with a *bad* init

We deliberately over-shrink the initial weights. This creates the vanishing-gradient pathology in its purest form.

In [ ]:
class DeepMLP(nn.Module):
    def __init__(self, d_in=64, hidden=128, depth=10, n_classes=2,
                 use_norm=False, init_scale=0.1):
        super().__init__()
        self.depth = depth
        self.use_norm = use_norm
        self.layers = nn.ModuleList()
        self.norms = nn.ModuleList()
        # Input projection
        self.layers.append(nn.Linear(d_in, hidden))
        self.norms.append(nn.LayerNorm(hidden) if use_norm else nn.Identity())
        # depth-1 hidden blocks
        for _ in range(depth - 1):
            self.layers.append(nn.Linear(hidden, hidden))
            self.norms.append(nn.LayerNorm(hidden) if use_norm else nn.Identity())
        self.head = nn.Linear(hidden, n_classes)
        # Deliberately tiny init — this is what causes layer-1 gradients to vanish.
        # A good init (Kaiming) would use gain / sqrt(fan_in); we intentionally go smaller.
        for m in self.layers:
            nn.init.normal_(m.weight, mean=0.0, std=init_scale)
            nn.init.zeros_(m.bias)
    def forward(self, x):
        for lin, norm in zip(self.layers, self.norms):
            x = norm(lin(x))
            # Tanh saturates at +-1 — its derivative dies when x is large, amplifying
            # the vanishing-gradient story. Using ReLU would hide the effect.
            x = torch.tanh(x)
        return self.head(x)

model_bad = DeepMLP(use_norm=False).to(device)
print(model_bad)
n_params = sum(p.numel() for p in model_bad.parameters())
print(f'param count: {n_params:,}')

### Step 3 — Per-layer gradient diagnostic

After one forward+backward pass, we measure the RMS of the gradient at each layer. In a healthy network these should all be the same order of magnitude; in a pathological one, layer 1 is several orders of magnitude smaller than layer 10.

In [ ]:
def per_layer_grad_rms(model, X, y):
    # Zero any stale grads, run one batch, measure ||grad||_2 / sqrt(numel) per linear layer.
    model.zero_grad()
    logits = model(X)
    loss = F.cross_entropy(logits, y)
    loss.backward()
    stats = []
    for i, layer in enumerate(model.layers):
        g = layer.weight.grad
        rms = g.pow(2).mean().sqrt().item()
        stats.append(rms)
    return loss.item(), stats

In [ ]:
loss_bad, grads_bad = per_layer_grad_rms(model_bad, X, y)
print(f'loss (untrained, no norm): {loss_bad:.4f}')
for i, g in enumerate(grads_bad):
    print(f'  layer {i:2d}  grad rms = {g:.2e}')

Notice how the earliest layers have gradients several orders of magnitude smaller than the last layers. With an optimizer step proportional to the gradient, the first layers effectively do not learn.

In [ ]:
model_norm = DeepMLP(use_norm=True).to(device)
loss_norm, grads_norm = per_layer_grad_rms(model_norm, X, y)
print(f'loss (untrained, with LayerNorm): {loss_norm:.4f}')
for i, g in enumerate(grads_norm):
    print(f'  layer {i:2d}  grad rms = {g:.2e}')

In [ ]:
# Plot the two profiles on a log axis — the vanishing-gradient effect jumps out.
plt.figure(figsize=(7, 4))
plt.plot(grads_bad, 'o-', label='no norm (vanishing)')
plt.plot(grads_norm, 's-', label='with LayerNorm')
plt.yscale('log')
plt.xlabel('layer index (0 = first)')
plt.ylabel('grad RMS (log)')
plt.title('Per-layer gradient magnitude, one batch, 10-layer MLP')
plt.legend(); plt.grid(alpha=0.3); plt.show()

## Experiment: train both variants end to end

Same optimizer, same data, same init seed. Only `use_norm` differs.

In [ ]:
def train(model, epochs=30, lr=1e-2):
    opt = torch.optim.SGD(model.parameters(), lr=lr)
    history = []
    for epoch in range(epochs):
        logits = model(X)
        loss = F.cross_entropy(logits, y)
        opt.zero_grad()
        loss.backward()
        opt.step()
        # Track accuracy too so we see real progress, not just loss.
        acc = (logits.argmax(-1) == y).float().mean().item()
        history.append((loss.item(), acc))
        if epoch % 5 == 0:
            print(f'  epoch {epoch:3d}  loss={loss.item():.4f}  acc={acc:.3f}')
    return history

In [ ]:
print('=== No norm ===')
torch.manual_seed(0)
h_bad = train(DeepMLP(use_norm=False).to(device))

print('\n=== With LayerNorm ===')
torch.manual_seed(0)
h_norm = train(DeepMLP(use_norm=True).to(device))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot([x[0] for x in h_bad], label='no norm')
ax[0].plot([x[0] for x in h_norm], label='LayerNorm')
ax[0].set_title('loss'); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].plot([x[1] for x in h_bad], label='no norm')
ax[1].plot([x[1] for x in h_norm], label='LayerNorm')
ax[1].set_title('accuracy'); ax[1].legend(); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Variations

### Mixed precision

`torch.amp.autocast` casts eligible ops to `float16`/`bfloat16`, halving memory bandwidth and cutting matmul time on tensor cores. `GradScaler` keeps fp16 gradients from underflowing. Below we time a batch of forward+backward in fp32 vs. mixed precision on the same (bigger) model.

In [ ]:
if device != 'cuda':
    print('Mixed-precision timing only makes sense on a GPU. Skipping.')
else:
    # Bigger model + bigger batch so wall-clock differences are measurable.
    big = DeepMLP(d_in=256, hidden=512, depth=12, use_norm=True).to(device)
    X_big = torch.randn(2048, 256, device=device)
    y_big = torch.randint(0, 2, (2048,), device=device)
    opt = torch.optim.Adam(big.parameters(), lr=1e-3)

    def time_fp32(iters=20):
        torch.cuda.synchronize()
        t0 = time.time()
        for _ in range(iters):
            opt.zero_grad()
            loss = F.cross_entropy(big(X_big), y_big)
            loss.backward()
            opt.step()
        torch.cuda.synchronize()
        return (time.time() - t0) / iters

    def time_amp(iters=20):
        scaler = torch.amp.GradScaler('cuda')
        torch.cuda.synchronize()
        t0 = time.time()
        for _ in range(iters):
            opt.zero_grad()
            with torch.amp.autocast('cuda', dtype=torch.float16):
                loss = F.cross_entropy(big(X_big), y_big)
            # GradScaler prevents fp16 gradients from silently rounding to 0.
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
        torch.cuda.synchronize()
        return (time.time() - t0) / iters

    # Warm up once so we do not measure kernel compile time.
    _ = time_fp32(iters=5)
    _ = time_amp(iters=5)
    t_fp32 = time_fp32()
    t_amp = time_amp()
    print(f'fp32 step: {t_fp32*1000:.2f} ms')
    print(f'amp  step: {t_amp*1000:.2f} ms')
    print(f'speedup:   {t_fp32 / t_amp:.2f}x')

## What to try next

- Swap `nn.LayerNorm` for `nn.BatchNorm1d` and compare the gradient histograms. BatchNorm fixes vanishing gradients too but introduces a train/eval mismatch LayerNorm avoids.
- Try Kaiming init (`nn.init.kaiming_normal_`) without any normalization. Does correct init alone fix the problem?
- Add residual connections (identity skip around every pair of layers) and re-run the per-layer gradient diagnostic. This is the ResNet story — covered in the next notebook.
- Replace tanh with ReLU and GELU and see how the per-layer RMS profile changes.